## ROI Extraction


ROI (Region of Interest) has been extracted from the real dataset.

In [2]:
import cv2
import mediapipe as mp
import numpy as np
import os
from pathlib import Path


In [7]:
TUNING_CONFIG = {
    
    "input_base_dir": Path(r"C:\Users\singh\Users\singh\Desktop\Palm detection\Research paper\01_data 5"),
    "output_roi_dir": Path(r"C:\Users\singh\Users\singh\Desktop\Palm detection\ROI\01_Symmetric_Palm_ROI_3"),
    
    
    "mp_min_detection_confidence": 0.3,  
    
   
    "roi_size_multiplier": 1.1,         
    "roi_center_shift": 0.5,            
    "target_resolution": (224, 224)     
}

In [8]:
# Ensure destination path exists
os.makedirs(TUNING_CONFIG["output_roi_dir"], exist_ok=True)

# Initialize Hand Landmark Detector
mp_hands = mp.solutions.hands
detector = mp_hands.Hands(
    static_image_mode=True, 
    max_num_hands=1, 
    min_detection_confidence=TUNING_CONFIG["mp_min_detection_confidence"]
)

In [9]:
# 2. ROI EXTRACTION WITH FALLBACK SAFETY

def extract_symmetric_palm(img):
    h, w, _ = img.shape
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = detector.process(img_rgb)
    
    
    # FALLBACK BACKUP: If MediaPipe fails, do a clean center crop
  
    if not results.multi_hand_landmarks: 
        # Calculate a square box right in the center of the image frame
        crop_size = min(h, w) // 2
        start_x = (w - crop_size) // 2
        start_y = (h - crop_size) // 2
        
        fallback_roi = img[start_y:start_y+crop_size, start_x:start_x+crop_size]
        roi_resized = cv2.resize(fallback_roi, TUNING_CONFIG["target_resolution"])
        
        # Default to "Unverified" subfolder so you know which images failed MediaPipe
        return roi_resized, "Unverified"

    
    # STANDARD PIPELINE: If MediaPipe succeeds, run your alignment
    
    lm = results.multi_hand_landmarks[0].landmark
    def get_pt(idx): return np.array([lm[idx].x * w, lm[idx].y * h], dtype="float32")

    p0 = get_pt(0)   # Wrist
    p5 = get_pt(5)   # Index Base
    p9 = get_pt(9)   # Middle Base
    p17 = get_pt(17) # Pinky Base

    # Geometric Handedness Determination via Cross Product
    v1 = p5 - p0
    v2 = p17 - p0
    det = v1[0] * v2[1] - v1[1] * v2[0]
    hand_label = "Right" if det > 0 else "Left"

    # RIGID VERTICAL HAND ALIGNMENT SYSTEM
    v_vertical = p9 - p0
    v_vertical_unit = v_vertical / np.linalg.norm(v_vertical)
    angle = np.arctan2(v_vertical_unit[1], v_vertical_unit[0]) - np.pi/2

    v_horizontal_unit = np.array([-v_vertical_unit[1], v_vertical_unit[0]], dtype="float32")
    palm_width = np.linalg.norm(p17 - p5)
    
    roi_center = p9 - (v_vertical_unit * (palm_width * TUNING_CONFIG["roi_center_shift"]))
    half_size = (palm_width * TUNING_CONFIG["roi_size_multiplier"]) / 2.0
    
    local_top_left     = roi_center - (v_horizontal_unit * half_size) + (v_vertical_unit * half_size)
    local_top_right    = roi_center + (v_horizontal_unit * half_size) + (v_vertical_unit * half_size)
    local_bottom_right = roi_center + (v_horizontal_unit * half_size) - (v_vertical_unit * half_size)
    local_bottom_left  = roi_center - (v_horizontal_unit * half_size) - (v_vertical_unit * half_size)
    
    src_pts = np.array([local_top_left, local_top_right, local_bottom_right, local_bottom_left], dtype="float32")
    
    target_w, target_h = TUNING_CONFIG["target_resolution"]
    dst_pts = np.array([[0, 0], [target_w, 0], [target_w, target_h], [0, target_h]], dtype="float32")

    # Background Masking Execution
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillConvexPoly(mask, src_pts.astype(int), 255)
    img_no_bg = cv2.bitwise_and(img, img, mask=mask)
    
    M = cv2.getPerspectiveTransform(src_pts, dst_pts)
    roi = cv2.warpPerspective(img_no_bg, M, TUNING_CONFIG["target_resolution"])
    
    return roi, hand_label


# 3. DATASET LOOP & SAVING PIPELINE

if __name__ == "__main__":
    valid_extensions = {'.jpg', '.jpeg', '.png', '.tiff', '.bmp'}
    image_paths = [
        p for p in TUNING_CONFIG["input_base_dir"].rglob('*') 
        if p.suffix.lower() in valid_extensions
    ]

    print(f"Found {len(image_paths)} images across folders. Processing...")
    
    processed_count = 0
    fallback_count = 0
    
    for img_path in image_paths:
        img = cv2.imread(str(img_path))
        if img is None: 
            print(f"Could not read image file: {img_path.name}")
            continue
        
        roi, hand_label = extract_symmetric_palm(img)
        
        if roi is not None:
            if hand_label == "Unverified":
                fallback_count += 1
            else:
                processed_count += 1
                
            user_id = img_path.parent.name
            save_dir = TUNING_CONFIG["output_roi_dir"] / user_id / hand_label
            os.makedirs(save_dir, exist_ok=True)
            
            save_path = save_dir / f"final_{img_path.stem}.png"
            cv2.imwrite(str(save_path), roi)

    print("\n" + "="*55)
    print(f"DATASET PROCESSING RUN COMPLETE")
    print(f"Successfully aligned by landmarks: {processed_count} images")
    print(f"Saved via Safety Center-Crop fallback: {fallback_count} images")
    print(f"Total files saved: {processed_count + fallback_count} / {len(image_paths)}")
    print(f"Target Save Directory: {TUNING_CONFIG['output_roi_dir']}")
    print("="*55)

Found 2288 images across folders. Processing...

DATASET PROCESSING RUN COMPLETE
Successfully aligned by landmarks: 2288 images
Saved via Safety Center-Crop fallback: 0 images
Total files saved: 2288 / 2288
Target Save Directory: C:\Users\singh\Users\singh\Desktop\Palm detection\ROI\01_Symmetric_Palm_ROI_3
